# ReefScan — Serving & Inference extras (Parts B–D)

Three GPU artifacts, meant to run on **Colab Pro (L4 or A100)**:
- **B** — a fused **CUDA preprocessing kernel** (`uint8 HWC → float32 NCHW + ImageNet-normalize`) in one memory pass
- **C** — self-hosted **Qwen2.5-VL-7B** via vLLM → re-run the VLM benchmark (adds a self-hosted column next to GPT-4o)
- **D** — a **vLLM serving throughput sweep** (concurrency 1→32)

**Runtime → Change runtime type → L4 (or A100)**, then run top-to-bottom. Every measured number is printed by its cell — nothing here is precomputed. Record the GPU shown in the first cell next to any number you keep.

## 0. Clone repo + confirm GPU

In [ ]:
!git clone -q https://github.com/HrishiKabra/reefscan.git 2>/dev/null || echo 'repo already present'
%cd /content/reefscan
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU, '| torch', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> L4/A100'

## B. Fused CUDA preprocessing kernel

The preprocessing *tail* — `uint8 HWC → float32 NCHW → ImageNet-normalize` — is normally 3–4 separate
CUDA kernels (cast, transpose, subtract, divide), each a full pass over memory. We fuse them into **one**
kernel that reads each `uint8` once and writes each `float32` once.

**Honest framing up front:** this is a **memory-bandwidth-bound** op. The win is real (fewer memory
round-trips) but small in *end-to-end* terms. The value is (a) knowing preprocessing's actual share of
latency *before* optimizing it, and (b) demonstrating writing, binding, and **verifying** a custom kernel.
So we profile the share first, then build → check correctness → benchmark.

### B1. Profile: what share of latency is preprocessing?

In [ ]:
import time
from transformers import AutoModel
dev = 'cuda'
model = AutoModel.from_pretrained('facebook/dinov2-base').to(dev).eval().half()
mean = torch.tensor([0.485,0.456,0.406], device=dev).view(1,3,1,1)
std  = torch.tensor([0.229,0.224,0.225], device=dev).view(1,3,1,1)
N = 32
u8 = torch.randint(0, 255, (N,224,224,3), dtype=torch.uint8, device=dev)

def preproc(x):  # the MULTI-OP torch tail (cast + transpose + scale + normalize)
    y = x.permute(0,3,1,2).contiguous().float().div_(255.0)
    return (y - mean) / std

def bench(fn, it=100, warm=20):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(it): fn()
    torch.cuda.synchronize(); return (time.perf_counter()-t)/it*1000

xp = preproc(u8).half()
pre_ms = bench(lambda: preproc(u8))
with torch.no_grad():
    fwd_ms = bench(lambda: model(xp))
share = pre_ms/(pre_ms+fwd_ms)*100
print(f'[{GPU}] batch={N}: preproc {pre_ms:.3f} ms | DINOv2-B fwd {fwd_ms:.3f} ms | '
      f'preproc = {share:.1f}% of the classify step')
print('e2e note: SAM2 AMG (~seconds) dominates the true pipeline, so preproc is a far smaller share end-to-end')

### B2. The kernel — build (load_inline), verify correctness, benchmark

In [ ]:
# nvcc must be on PATH to compile the extension; verbose=True surfaces any compiler error
import shutil
!which nvcc >/dev/null 2>&1 && nvcc --version | tail -1 || echo '!! no nvcc — run: !apt-get -qq install -y cuda-toolkit-12-8'
!pip -q install ninja
shutil.rmtree('/root/.cache/torch_extensions', ignore_errors=True)
from torch.utils.cpp_extension import load_inline
cpp_src = 'torch::Tensor fused_preproc(torch::Tensor img, torch::Tensor mean, torch::Tensor std);'
cuda_src = r'''
#include <torch/extension.h>
#include <cuda_runtime.h>
#include <cstdint>

// One thread per element. Reads each uint8 once (HWC), writes each float32 once (CHW).
// Fuses: cast(uint8->f32) + /255 + transpose(HWC->CHW) + normalize((x-mean)/std).
__global__ void fused_preproc_kernel(const uint8_t* __restrict__ in,
                                     float* __restrict__ out,
                                     const float* __restrict__ mean,
                                     const float* __restrict__ std,
                                     int N, int H, int W) {
    long idx = (long)blockIdx.x * blockDim.x + threadIdx.x;
    long total = (long)N * H * W * 3;
    if (idx >= total) return;
    int  c  = idx % 3;          // input is N,H,W,C contiguous
    long hw = idx / 3;
    int  w  = hw % W;
    long nh = hw / W;
    int  h  = nh % H;
    int  n  = nh / H;
    float v = (float)in[idx] / 255.0f;
    v = (v - mean[c]) / std[c];
    long out_idx = (((long)n * 3 + c) * H + h) * W + w;   // write N,C,H,W
    out[out_idx] = v;
}

torch::Tensor fused_preproc(torch::Tensor img, torch::Tensor mean, torch::Tensor std) {
    TORCH_CHECK(img.is_cuda() && img.dtype() == torch::kUInt8, "img must be cuda uint8 [N,H,W,3]");
    TORCH_CHECK(img.is_contiguous(), "img must be contiguous");
    int N = img.size(0), H = img.size(1), W = img.size(2);
    auto out = torch::empty({N, 3, H, W},
        torch::TensorOptions().dtype(torch::kFloat32).device(img.device()));
    long total = (long)N * H * W * 3;
    int threads = 256;
    long blocks = (total + threads - 1) / threads;
    fused_preproc_kernel<<<blocks, threads>>>(
        img.data_ptr<uint8_t>(), out.data_ptr<float>(),
        mean.data_ptr<float>(), std.data_ptr<float>(), N, H, W);
    return out;
}
'''
mod = load_inline(name='reefscan_preproc', cpp_sources=cpp_src, cuda_sources=cuda_src,
                  functions=['fused_preproc'], verbose=True, extra_cuda_cflags=['-O2'])
print('kernel built OK')

mean1 = torch.tensor([0.485,0.456,0.406], device='cuda')
std1  = torch.tensor([0.229,0.224,0.225], device='cuda')
out_k   = mod.fused_preproc(u8, mean1, std1)
out_ref = preproc(u8)
diff = (out_k - out_ref).abs().max().item()
print('max abs diff  kernel vs multi-op torch:', diff)
assert torch.allclose(out_k, out_ref, atol=1e-4), 'kernel output mismatch!'
print('correctness OK (matches the multi-op torch pipeline)')

k_ms = bench(lambda: mod.fused_preproc(u8, mean1, std1))
r_ms = bench(lambda: preproc(u8))
bytes_moved = N*224*224*3*(1+4)   # read uint8 + write float32
print(f'[{GPU}] fused kernel {k_ms:.4f} ms | multi-op torch {r_ms:.4f} ms | '
      f'speedup {r_ms/k_ms:.2f}x | kernel effective BW {bytes_moved/k_ms/1e6:.0f} GB/s')

**B result (fill from the run above, GPU = the name printed in cell 0):** the fused kernel matches the
multi-op torch tail to < 1e-4 and is ~`<speedup>`× faster by collapsing 4 passes into 1. It runs near the
card's memory bandwidth (`<GB/s>`), which is the ceiling for this op — so this is a bandwidth win, not a
compute one. Because preprocessing is only ~`<share>`% of the classify step (and far less end-to-end,
where SAM2 dominates), the honest takeaway is: *small e2e gain, real demonstration of kernel authoring +
binding + verification.*

## C. Self-hosted VLM baseline via vLLM (Qwen2.5-VL-7B)

Replace the paid GPT-4o VLM baseline with a **self-hosted** open VLM served by vLLM's OpenAI-compatible
server, and re-run the exact same benchmark (forced A/B + logprobs → accuracy, macro-F1, ECE) on the same
NOAA test split. **GPT-4o stays as a column; we ADD Qwen.** The coral patches are 224px crops sent
unresized — Qwen2.5-VL sees them at native resolution.

Needs ~16 GB VRAM in fp16 (L4/A100 fine). Pinned to **vLLM 0.8.5** (Qwen2.5-VL support + `vllm bench serve`).

In [ ]:
!pip install -q vllm==0.8.5
print('>>> If the SERVER cell below errors with a CUDA/transformers ABI mismatch, do '
      'Runtime -> Restart session, then re-run from THIS cell down (the git clone persists).')

### C1. Start the vLLM OpenAI-compatible server (backgrounded)

In [ ]:
import subprocess, sys, time, os, urllib.request
os.makedirs('logs', exist_ok=True)
srv = subprocess.Popen(
    [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
     '--model', 'Qwen/Qwen2.5-VL-7B-Instruct',
     '--gpu-memory-utilization', '0.9',
     '--max-model-len', '8192',      # leaves KV-cache headroom for higher concurrency
     '--max-logprobs', '20',         # A/B logprobs need top_logprobs <= this
     '--limit-mm-per-prompt', 'image=1',
     '--port', '8000'],
    stdout=open('logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('starting vLLM (first run downloads ~16GB weights + warms up — can take several minutes)...')
up = False
for i in range(180):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); up = True; break
    except Exception: time.sleep(5)
print('vLLM server UP after ~%ds' % (i*5) if up else 'NOT up yet — inspect logs/vllm.log')
!tail -6 logs/vllm.log

### C2. Run the benchmark against the local server
Full test set (1,565 imgs) is resumable (JSONL cache) — start with `--limit 300` for a fast first pass,
then drop `--limit` for the full number. `--workers 16` drives concurrency at the server.

In [ ]:
!python -m backend.vlm_benchmark --model Qwen/Qwen2.5-VL-7B-Instruct \
    --base-url http://localhost:8000/v1 --api-key EMPTY --workers 16 --limit 300

### C3. Three-way comparison — DINOv2 specialist vs GPT-4o vs Qwen2.5-VL

In [ ]:
import json, glob
from pathlib import Path
E = Path('docs/eval')
bym = {}
spec = json.loads((E/'metrics.json').read_text())
bym['DINOv2-B specialist'] = (spec['accuracy'], spec['macro_f1'], spec.get('ece'))
# old committed gpt-4o summary + any new per-model files
cands = list(glob.glob(str(E/'vlm_benchmark_*.json')))
if (E/'vlm_benchmark.json').exists(): cands.append(str(E/'vlm_benchmark.json'))
for f in cands:
    d = json.loads(open(f).read())
    bym[d['model']] = (d['accuracy'], d['macro_f1'], d.get('ece'))
print(f"{'model':<32}{'accuracy':>9}{'macro-F1':>10}{'ECE':>8}")
print('-'*59)
for m,(a,f1,e) in bym.items():
    es = f'{e:.4f}' if e is not None else '  n/a'
    print(f'{m:<32}{a:>9.4f}{f1:>10.4f}{es:>8}')

**C result (fill from the run):** DINOv2-B specialist `<acc/F1/ECE>` vs zero-shot GPT-4o `<...>` vs
self-hosted Qwen2.5-VL-7B `<...>`. Expectation: the tiny fine-tuned specialist beats both generalist VLMs
on accuracy + calibration, and the self-hosted Qwen is a $0, reproducible stand-in for the paid GPT-4o
baseline (same prompt, same test split).

## D. vLLM serving throughput sweep (concurrency 1→32)

The serving curve for the self-hosted VLM: p50/p95/p99 latency + throughput as concurrency rises.
Primary path uses vLLM's built-in **`vllm bench serve`**; if the CLI/keys differ in your vLLM build, the
cell falls back to a small async sweep of the *actual* image workload against the chat endpoint so you
always get numbers. (No Docker/Triton in Colab — that's the RunPod appendix.)

In [ ]:
import json, subprocess, asyncio, io, time
import numpy as np
from PIL import Image
import httpx
LEVELS = [1,4,8,16,32]

def _pctile(v,p):
    if not v: return 0.0
    v=sorted(v); k=(len(v)-1)*p/100; f=int(k)
    return round(v[f]+(v[min(f+1,len(v)-1)]-v[f])*(k-f),1)

rows = []
# --- primary: vllm bench serve (text 'random' dataset; measures decode/serving throughput) ---
for c in LEVELS:
    rf = f'logs/bench_c{c}.json'
    subprocess.run(['vllm','bench','serve','--backend','openai-chat',
        '--model','Qwen/Qwen2.5-VL-7B-Instruct','--base-url','http://localhost:8000',
        '--endpoint','/v1/chat/completions','--dataset-name','random','--num-prompts','100',
        '--max-concurrency',str(c),'--percentile-metrics','e2el','--metric-percentiles','50,95,99',
        '--save-result','--result-filename',rf], check=False)
    try:
        d=json.load(open(rf))
        rows.append({'concurrency':c,'p50':d.get('median_e2el_ms') or d.get('p50_e2el_ms'),
                     'p95':d.get('p95_e2el_ms'),'p99':d.get('p99_e2el_ms'),
                     'throughput':d.get('request_throughput')})
    except Exception as ex:
        print('vllm bench serve produced no parseable result for c=%d (%s)'%(c,ex))

# --- fallback: async sweep of the real image workload if the CLI path yielded nothing ---
if not rows:
    print('falling back to async image-workload sweep...')
    arr=np.random.default_rng(0).integers(0,255,(224,224,3),dtype=np.uint8)
    buf=io.BytesIO(); Image.fromarray(arr).save(buf,format='PNG')
    import base64; b64=base64.b64encode(buf.getvalue()).decode()
    body=lambda: {'model':'Qwen/Qwen2.5-VL-7B-Instruct','max_tokens':1,'temperature':0,
        'messages':[{'role':'user','content':[{'type':'text','text':'A for healthy, B for bleached. One letter.'},
        {'type':'image_url','image_url':{'url':'data:image/png;base64,'+b64}}]}]}
    async def one(cl):
        t=time.perf_counter(); r=await cl.post('http://localhost:8000/v1/chat/completions',json=body()); r.raise_for_status()
        return (time.perf_counter()-t)*1000
    async def level(c,n=48):
        sem=asyncio.Semaphore(c); lat=[]
        async with httpx.AsyncClient(timeout=120) as cl:
            async def w():
                async with sem: lat.append(await one(cl))
            t=time.perf_counter(); await asyncio.gather(*[w() for _ in range(n)]); wall=time.perf_counter()-t
        return {'concurrency':c,'p50':_pctile(lat,50),'p95':_pctile(lat,95),'p99':_pctile(lat,99),
                'throughput':round(n/wall,2)}
    for c in LEVELS: rows.append(await level(c))

print(f"[{GPU}]  conc   p50 ms   p95 ms   p99 ms   req/s")
for r in rows:
    print(f"      {r['concurrency']:>4} {r['p50']:>8} {r['p95']:>8} {r['p99']:>8} {r['throughput']:>7}")
json.dump({'gpu':GPU,'rows':rows}, open('docs/eval/vllm_serving_sweep.json','w'), indent=2)

In [ ]:
import matplotlib.pyplot as plt
cs=[r['concurrency'] for r in rows]
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4.5))
for k,col in [('p50','#2166ac'),('p95','#b2182b'),('p99','#762a83')]:
    a1.plot(cs,[r[k] for r in rows],'-o',label=k,color=col)
a1.set_xlabel('concurrency'); a1.set_ylabel('latency (ms)'); a1.set_title(f'Qwen2.5-VL serving latency ({GPU})')
a1.legend(); a1.grid(alpha=.25)
a2.plot(cs,[r['throughput'] for r in rows],'-o',color='#1b7837')
a2.set_xlabel('concurrency'); a2.set_ylabel('throughput (req/s)'); a2.set_title('Throughput'); a2.grid(alpha=.25)
fig.tight_layout(); fig.savefig('docs/eval/vllm_serving_sweep.png',dpi=140,bbox_inches='tight'); plt.show()

**D result (fill from the run):** p99 pulls away from p50 as concurrency rises and throughput saturates —
the serving curve for a self-hosted 7B VLM on `<GPU>`. Download `docs/eval/vllm_serving_sweep.{json,png}`
and paste the table back to commit it.